In [0]:
from pyspark.sql import functions as F

PATH = "/Volumes/voebem/bronze/arquivos/referencias"
TABLE_AERODROMES = "voebem.bronze.aerodromos"
TABLE_FOREIGN_COMPANIES = "voebem.bronze.foreign_companies"
TABLE_NATIONAL_COMPANIES = "voebem.bronze.national_companies"

UNQUOTED = chr(0)

In [0]:
raw_aerodrome = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", True)
    .option("skipRows", 1)
    .option("quote", UNQUOTED)
    .option("escape", '"')
    .option("encoding", "ISO-8859-1")
    .load(f"{PATH}/AerodromosPublicos.csv")
)

print("Columns read from file: ")
for col in raw_aerodrome.columns:
    print(f" {col!r}")

In [0]:
RENAME = {
    "Código OACI": "icao",
    "CIAD": "ciad",
    "Nome": "nome",
    "Município": "municipio",
    "UF": "uf",
    "Município Servido": "municipio_servido",
    "UF Servido": "uf_servido",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Altitude": "altitude",
    "Situação": "situacao",
}

missing = [col for col in RENAME if col not in raw_aerodrome.columns]
assert not missing, f"Missing columns: {missing} not found in file"

renamed = raw_aerodrome.select(
    *[F.col(f"`{origin}`").cast("string").alias(new) for origin,
    new in RENAME.items()]
)

In [0]:
bronze_aerodromes = (
    renamed
    .withColumn("_arquivo_origem", F.col("_metadata.file_name"))
    .withColumn("_ingerido_em", F.current_timestamp())
)

In [0]:
(
    bronze_aerodromes.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TABLE_AERODROMES)
)

print(f"{TABLE_AERODROMES}: {spark.table(TABLE_AERODROMES).count():,} rows")

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {TABLE_AERODROMES} IS
    'Bronze - cadastro de aerodromos publicos da ANAC, como chegou. Chave: codigo ICAO (OACI). 
    Cobre apenas aerodromos brasileiros - aeroportos estrangeiros do VRA nao estao aqui.'
""")

In [0]:
def read_companies(file: str):
    return (
        spark.read.format("csv")
        .option("sep", ";")
        .option("header", "true")
        .option("skipRows", 1)
        .option("encoding", "UTF-8")
        .option("quote", '"')
        .load(f"{PATH}/{file}")
        .select(
            F.col("ICAO").alias("icao"),
            F.col("Estrangeira").alias("sigla_iata"),
            F.col("Razao").alias("razao_social"),
            F.col("Servico").alias("servico"),
            F.col("Cidade").alias("cidade"),
            F.col("UF").alias("uf"),
            F.col("Ativa").alias("situacao"),
        )
        .withColumn("_arquivo_origem", F.lit(file))
        .withColumn("_ingerido_em", F.current_timestamp())
    )


for file, table in [
    ("pda_empresas_aereas_nacionais.csv",    TABLE_NATIONAL_COMPANIES),
    ("pda_empresas_aereas_estrangeiros.csv", TABLE_FOREIGN_COMPANIES),
]:
    (
        read_companies(file).write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(table)
    )
    print(f"{table}: {spark.table(table).count():,} linhas")


In [0]:
FLIGHT_OPERATION_CODES = [
    ("codigo_di", "0", "Etapa Regular"),
    ("codigo_di", "2", "Etapa Extra"),
    ("codigo_di", "3", "Etapa de Retorno"),
    ("codigo_di", "4", "Inclusão de Etapa"),
    ("codigo_di", "6", "Etapa Não Remunerada Sem Transporte de Objetos"),
    ("codigo_di", "7", "Etapa de Voo de Fretamento"),
    ("codigo_di", "9", "Etapa de Voo Charter"),
    ("codigo_di", "D", "Etapa de Voo Duplicada"),
    ("codigo_di", "E", "Etapa Não Remunerada Com Transporte de Objetos"),
    ("codigo_tipo_linha", "N", "Doméstica Mista"),
    ("codigo_tipo_linha", "C", "Doméstica Cargueira"),
    ("codigo_tipo_linha", "I", "Internacional Mista"),
    ("codigo_tipo_linha", "G", "Internacional Cargueira"),
]

flight_operation_codes = spark.createDataFrame(FLIGHT_OPERATION_CODES, "dominio string, codigo string, descricao string")

(
    flight_operation_codes.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voebem.bronze.codigos_operacao")
)

print(f"bronze.flight_operation_codes: {spark.table('voebem.bronze.codigos_operacao').count()} rows")
display(spark.table("voebem.bronze.codigos_operacao"))


In [0]:
for table, comment in [
    ("voebem.bronze.empresas_nacionais",
     "Bronze - cadastro de empresas aereas NACIONAIS da ANAC, como chegou. Chave: codigo ICAO. "
     "Nao unir com empresas_estrangeiras nesta camada: a uniao e feita na silver."),
    ("voebem.bronze.empresas_estrangeiras",
     "Bronze - cadastro de empresas aereas ESTRANGEIRAS autorizadas a operar no Brasil, como chegou. "
     "Chave: codigo ICAO. Cadastro separado do nacional na origem, mantido separado no bronze."),
    ("voebem.bronze.codigos_operacao",
     "Bronze - seed table curada a partir da pagina de descricao de variaveis da ANAC. "
     "Traduz codigo_di e codigo_tipo_linha para descricao em portugues."),
]:
    spark.sql(f"COMMENT ON TABLE {table} IS '{comment}'")

print("comments applied")